In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as e:
    print("Could not read transformers version:", e)

Torch version: 2.11.0
Transformers version: 5.3.0


In [3]:
SPLITS = Path("../01_data/interim/splits")
TABLES = Path("../04_outputs/tables")
CHECKPOINTS = Path("../03_models/checkpoints")

TABLES.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
EPSILON = 0.5


In [ ]:
train_df = pd.read_csv(SPLITS / "banglasarc3_binary_train.csv")
val_df = pd.read_csv(SPLITS / "banglasarc3_binary_val.csv")
test_df = pd.read_csv(SPLITS / "banglasarc3_binary_test.csv")

(20508, 4) (2564, 4) (2564, 4)


,text,label_binary,label_original,dataset_name
0,আমার ভাই এবং বাবাকে নিয়ে নােয়াখালী সদর থানায...,0,0,ben_sarc
1,যদি ডট বল করাতে চাও তবে আমাকে ডাকো সুনীল নারিন...,0,0,ben_sarc
2,কেউ অক্সিজেন না পেয়ে মরে আর কেউ বিয়ে করে মরে ।,1,1,ben_sarc
3,সেইদিন তোরা আম্পায়ারদের সাথে আঁতাত করে টাকা আর...,0,0,ben_sarc
4,একটা বাচ্চা জন্মের পর থেকেই বুলিং শুরু হয়ে যায়...,0,0,ben_sarc


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
train_df = train_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
val_df = val_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
test_df = test_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})

In [7]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [8]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

Map: 100%|██████████| 2564/2564 [00:00<00:00, 16418.52 examples/s]


In [9]:
train_ds = train_ds.remove_columns(["text"])
val_ds = val_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

In [11]:
class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

for name, param in model.named_parameters():
    if "embedding" in name.lower():
        print(name)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 50879.18it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

electra.embeddings.word_embeddings.weight
electra.embeddings.position_embeddings.weight
electra.embeddings.token_type_embeddings.weight
electra.embeddings.LayerNorm.weight
electra.embeddings.LayerNorm.bias


In [13]:
EMB_NAME = "word_embeddings"

In [14]:
class FGMTrainer(Trainer):
    def __init__(self, *args, epsilon=0.5, emb_name="word_embeddings", **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model, epsilon=epsilon, emb_name=emb_name)

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        outputs = model(**inputs)
        loss = outputs.loss

        self.accelerator.backward(loss)

        self.fgm.attack()
        outputs_adv = model(**inputs)
        loss_adv = outputs_adv.loss
        self.accelerator.backward(loss_adv)
        self.fgm.restore()

        return loss.detach() / self.args.gradient_accumulation_steps

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 51156.38it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [ ]:
training_args = TrainingArguments(
    output_dir="../03_models/checkpoints/banglabert_fgm_banglasarc3_binary",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

In [17]:
trainer = FGMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    epsilon=EPSILON,
    emb_name=EMB_NAME,
)

In [18]:
trainer.train()

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Binary,Recall Binary,F1 Binary,Macro F1
1,0.506111,0.469582,0.790952,0.817177,0.749610,0.781937,0.790594
2,0.335644,0.457936,0.803822,0.804059,0.803432,0.803746,0.803822


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.74it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.Lay

TrainOutput(global_step=5128, training_loss=0.4208772513498196, metrics={'train_runtime': 7328.826, 'train_samples_per_second': 5.597, 'train_steps_per_second': 0.7, 'total_flos': 2697940761661440.0, 'train_loss': 0.4208772513498196, 'epoch': 2.0})

In [19]:
def predict_metrics(trainer, df_for_labels, ds):
    output = trainer.predict(ds)
    preds = np.argmax(output.predictions, axis=-1)
    labels = np.array(df_for_labels["label"])

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    metrics = {
        "accuracy": float(acc),
        "precision_binary": float(p_bin),
        "recall_binary": float(r_bin),
        "f1_binary": float(f1_bin),
        "macro_f1": float(f1_macro),
    }
    return metrics, cm

val_metrics, val_cm = predict_metrics(trainer, val_df, val_ds)
test_metrics, test_cm = predict_metrics(trainer, test_df, test_ds)

print("Validation metrics:", val_metrics)
print("Validation confusion matrix:\n", val_cm)

print("\nTest metrics:", test_metrics)
print("Test confusion matrix:\n", test_cm)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'accuracy': 0.8038221528861155, 'precision_binary': 0.8040593286494926, 'recall_binary': 0.8034321372854915, 'f1_binary': 0.8037456106125634, 'macro_f1': 0.8038221230450732}
Validation confusion matrix:
 [[1031  251]
 [ 252 1030]]

Test metrics: {'accuracy': 0.8096723868954758, 'precision_binary': 0.8211974110032363, 'recall_binary': 0.7917316692667706, 'f1_binary': 0.8061953931691819, 'macro_f1': 0.8096111065462768}
Test confusion matrix:
 [[1061  221]
 [ 267 1015]]


In [ ]:
results = [
    {
        "model": "banglabert_fgm",
        "dataset": "banglasarc3_binary",
        "split": "validation",
        "accuracy": val_metrics["accuracy"],
        "precision_binary": val_metrics["precision_binary"],
        "recall_binary": val_metrics["recall_binary"],
        "f1_binary": val_metrics["f1_binary"],
        "macro_f1": val_metrics["macro_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "epsilon": EPSILON,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    },
    {
        "model": "banglabert_fgm",
        "dataset": "banglasarc3_binary",
        "split": "test",
        "accuracy": test_metrics["accuracy"],
        "precision_binary": test_metrics["precision_binary"],
        "recall_binary": test_metrics["recall_binary"],
        "f1_binary": test_metrics["f1_binary"],
        "macro_f1": test_metrics["macro_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "epsilon": EPSILON,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    },
]

results_df = pd.DataFrame(results)
results_df.to_csv(TABLES / "banglabert_fgm_banglasarc3_binary_results.csv", index=False)

with open(TABLES / "banglabert_fgm_banglasarc3_binary_confusion_matrices.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "validation": val_cm.tolist(),
            "test": test_cm.tolist(),
        },
        f,
        ensure_ascii=False,
        indent=2
    )

results_df

,model,dataset,split,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,epochs,batch_size,learning_rate,epsilon,max_length,seed
0,banglabert_fgm,ben_sarc_binary,validation,0.803822,0.804059,0.803432,0.803746,0.803822,2,8,0.00002,0.5,128,42
1,banglabert_fgm,ben_sarc_binary,test,0.809672,0.821197,0.791732,0.806195,0.809611,2,8,0.00002,0.5,128,42
